In [55]:
import pandas as pd
import plotly.express as px
from transformers import AutoTokenizer
import numpy as np
from pathlib import Path
import pickle
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
raw_csv_path = "../data/raw/AI_Human.csv"
df = pd.read_csv(raw_csv_path)

In [3]:
total_len = len(df)
total_ai = (df["generated"] == 1).sum()
total_human = (df["generated"] == 0).sum()

print(f"The dataset consists of {total_len} samples, of which {total_ai} are ai generated and {total_human} are human written.")

The dataset consists of 487235 samples, of which 181438 are ai generated and 305797 are human written.


In [4]:
df["text_len"] = df["text"].apply(lambda x: len(x))
df["word_count"] = df["text"].str.count(" ") + 1
median_char = df["text_len"].median().item()
median_word = df["word_count"].median().item()
print(f"Median word count equals {median_word} while median character count {median_char}")

Median word count equals 358.0 while median character count 2102.0


In [5]:
fig = px.histogram(
    df["text_len"],
    labels={
        "value": "Length of a specific entry",
        "count": "Count",
    },
)

fig

In [19]:
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")


def tokenize_function(examples):
    return tokenizer(
        examples,
        padding="longest",
    )


df_subset = df.sample(10_000)
tokenized = tokenize_function(list(df_subset["text"].to_numpy()))
tokenized = np.array(tokenized["attention_mask"])
df_subset["token_size"] = tokenized.sum(axis=1)
df_subset["truncated"] = df_subset["token_size"] > 512

fig = px.scatter(
    df_subset,
    "text_len",
    "token_size",
    color="truncated",
    opacity=0.1,
    labels={
        "token_size": "Number of generated tokens",
        "text_len": "Length of the text",
        "truncated": "Is output truncated",
    },
)
fig

Token indices sequence length is longer than the specified maximum sequence length for this model (1829 > 512). Running this sequence through the model will result in indexing errors


In [ ]:
df_subset["pred_token_size"] = df_subset["text_len"] / 5
tokens_lost = (df_subset["pred_token_size"] - df_subset["token_size"]).sum()# * len(df)/10_000
tokens_kept = df_subset["token_size"].sum()
tokens_total_pred = df_subset["pred_token_size"].sum()
truncate_loss = tokens_kept / (tokens_kept + tokens_lost)

print(f"We loose about {truncate_loss}")


We loose about 0.9012790039625183


In [26]:
model_dir = "models"


model_files = [file for file in Path.iterdir(Path(f"../{model_dir}")) if not Path.is_dir(file)]

model_dict = {}
for file in model_files:
    with open(file, "rb") as f:
        model_dict[file] = pickle.load(f)
model_dict

def get_feature_importances(model):
    if hasattr(model, "coef_"):
        return model.coef_.ravel()
    elif hasattr(model, "feature_importances_"):
        return model.feature_importances_


df_features = (
    pd.DataFrame(
        {
            model_path.stem: get_feature_importances(model)
            for model_path, model in model_dict.items()
        }
    )
    .reset_index(drop=False)
    .rename(columns={"index": "feature_num"})
)
df_features.head()

,feature_num,linear_svc,sgd_classifier,random_forest,logistic_regression,decision_tree
0,0,-0.137768,0.309068,0.000692,-0.040862,0.000329
1,1,0.132761,0.761442,0.000527,0.657918,0.000317
2,2,-0.124179,-1.096989,0.027710,-0.490668,0.016137
3,3,0.338633,0.778045,0.001486,0.950695,0.001738
4,4,-0.188860,-0.608605,0.000789,-0.574058,0.000887


## Model Correlation Analysis

**Two complementary analyses:**

### 1. Feature-Level Correlation
- Correlates model **weights/importances** across 768 BERT dimensions
- **Question:** "Do models rely on the same embedding dimensions?"

In [27]:
df_corr = df_features.drop(columns="feature_num").corr(method="spearman")
fig = px.imshow(
    df_corr,
    color_continuous_midpoint=0,
    color_continuous_scale="RdBu",
)
fig.data[0].text = df_corr.round(3).to_numpy()
fig.data[0].texttemplate = "%{text}"
fig.update_traces(textfont_size=18)
fig

### 2. Prediction-Level Correlation
- Correlates actual **predictions** across 10K test samples
- **Question:** "Do models agree on which specific texts are AI/Human?"
- Split by true label: Agreement on AI samples vs Human samples

In [64]:
# Option 2: Prediction-level correlation (sample-by-sample agreement)
from datasets import load_from_disk

dataset = load_from_disk("../data/processed/encoded_dataset/test")
X_test = np.array(dataset['embeddings'])
y_test = np.array(dataset['labels'])


y_data = {"ground_truth": y_test}
for model_path, model in model_dict.items():
    model_name = model_path.stem
    y_data[model_name] = model.predict(X_test)
    accuracy = (y_data[model_name] == y_test).mean()

pred_df = pd.DataFrame(y_data)

np_pred = pred_df.to_numpy(dtype=np.bool)

agreement_matrix = pred_df.T.dot(pred_df) + (1 - pred_df).T.dot(1 - pred_df)

agreement_df = pd.DataFrame(
    agreement_matrix / len(pred_df),
    index=pred_df.columns,
    columns=pred_df.columns,
)

fig = px.imshow(
    agreement_df,
)
fig.data[0].text = agreement_df.round(3).to_numpy()
fig.data[0].texttemplate = "%{text}"
fig.update_traces(textfont_size=18)
fig